Short-Term Indoor CO2 Prediction for Ventilation Monitoring

In [ ]:
from pathlib import Path
import json
import pandas as pd
try:
    import torch
except ImportError:
    torch = None

if torch is not None and torch.cuda.is_available():
    device = torch.device("cuda")
elif torch is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu") if torch is not None else "cpu"

print("Using device:", device)

In [ ]:
DATA_FOLDER = "MMA3001 Dataset - MSI Occupancy and Environmental"
search_locations = [Path.cwd(), *Path.cwd().parents]
data_dir = next((p / DATA_FOLDER for p in search_locations if (p / DATA_FOLDER).is_dir()), None)
if data_dir is None:
    raise FileNotFoundError(f"Could not find {DATA_FOLDER!r} from {Path.cwd()}")

occupancy_path = data_dir / "5occupancySensor_MayToDec2024_9MRows.csv"
environment_path = data_dir / "5EnvSensor_MayToDec2024_180kRows.csv"
locations_path = data_dir / "Sensor ID and Locations.xlsx"

# Set to a row count (e.g. 10_000) for a quick preview; None loads all rows.
NROWS = None
LOAD_OCCUPANCY = False  # Deferred until room mapping is resolved.
occupancy = pd.read_csv(
    occupancy_path, nrows=NROWS,
    dtype={"deviceid": "string", "floorspaceid": "string"},
) if LOAD_OCCUPANCY else None
environment = pd.read_csv(
    environment_path, nrows=NROWS, dtype={"sensorid": "string"},
)
# UTC keeps timestamps comparable across daylight-saving offsets.
for column in ["collecteddate", "occupancystatuschangedate"]:
    if occupancy is not None:
        occupancy[column] = pd.to_datetime(occupancy[column], format="mixed", utc=True)
for column in ["createdate", "processdate"]:
    environment[column] = pd.to_datetime(environment[column], format="mixed", utc=True)

# Keep both workbook sheets, including the occupancy sheet's original warning.
try:
    location_sheets = pd.read_excel(locations_path, sheet_name=None, dtype="string")
except ImportError:
    location_sheets = {}
    print("Location workbook skipped. To enable it, run %pip install openpyxl in this kernel.")
locations = location_sheets.get("ENV", pd.DataFrame())
occupancy_locations = location_sheets.get("OCC (Incorrect Data - SpaceFloo", pd.DataFrame())

print(f"Data folder: {data_dir}")
print(f"Occupancy: {occupancy.shape if occupancy is not None else 'deferred'}")
print(f"Environment: {environment.shape}")
for sheet_name, sheet in location_sheets.items():
    print(f"Locations [{sheet_name}]: {sheet.shape}")
display(environment.head(), locations.head())

## Audit environmental measurements
Use the measurement times inside JSON, rather than database creation times. Remove exact repeated measurements; exclude conflicting values at the same sensor/time/variable instead of choosing one arbitrarily. No interpolation or target filling is used. Nonpositive CO2 is excluded; other unusual values remain for inspection.

In [ ]:
from time import perf_counter
import numpy as np
started = perf_counter()
records, parse_errors = [], []
for row in environment.itertuples(index=False):
    try:
        payload = json.loads(row.jsondata)
        for item in payload:
            name = item.get("variable", {}).get("name")
            if name not in ["Carbon dioxide", "Temperature", "Humidity"]:
                continue
            for reading in item.get("values", []):
                records.append((row.sensorid, reading.get("time"), name, reading.get("value")))
    except (ValueError, TypeError, AttributeError) as exc:
        parse_errors.append((row.id, str(exc)))
readings = pd.DataFrame(records, columns=["sensorid", "timestamp", "variable", "value"])
readings["timestamp"] = pd.to_datetime(pd.to_numeric(readings["timestamp"], errors="coerce"), unit="s", utc=True, errors="coerce")
readings["value"] = pd.to_numeric(readings["value"], errors="coerce")
invalid = readings[["sensorid", "timestamp", "variable", "value"]].isna().any(axis=1) | ~np.isfinite(readings["value"])
invalid_count = int(invalid.sum())
readings = readings.loc[~invalid].copy()
exact_duplicates = int(readings.duplicated().sum())
readings = readings.drop_duplicates()
keys = ["sensorid", "timestamp", "variable"]
conflicts = readings.duplicated(keys, keep=False)
conflicting_readings = readings.loc[conflicts].copy()
readings = readings.loc[~conflicts]
environment_wide = readings.pivot(index=["sensorid", "timestamp"], columns="variable", values="value").reset_index()
co2 = environment_wide.rename(columns={"Carbon dioxide": "co2_ppm"}).sort_values(["sensorid", "timestamp"])
nonpositive_co2 = int(co2["co2_ppm"].le(0).sum())
co2 = co2.loc[co2["co2_ppm"].gt(0)].copy()
print({"JSON errors": len(parse_errors), "invalid readings": invalid_count,
       "exact duplicates removed": exact_duplicates, "conflicting readings excluded": len(conflicting_readings),
       "nonpositive CO2 excluded": nonpositive_co2})
print(f"Parsing and cleaning: {perf_counter() - started:.2f} seconds")
display(co2.head(), co2[["co2_ppm", "Temperature", "Humidity"]].describe())

In [ ]:
coverage_rows = []
for sensor, frame in co2.groupby("sensorid"):
    times = frame["timestamp"].sort_values()
    intervals = times.diff().dt.total_seconds().dropna()
    typical = intervals.median()
    coverage_rows.append({"sensorid": sensor, "observations": len(times),
        "start": times.min(), "end": times.max(), "median_interval_seconds": typical,
        "p95_interval_seconds": intervals.quantile(.95),
        "gaps_over_30min": int((intervals > 1800).sum()),
        "largest_gap_hours": intervals.max() / 3600})
sensor_audit = pd.DataFrame(coverage_rows).sort_values("observations", ascending=False)
display(sensor_audit)
# Identify contiguous periods using an explicit 30-minute outage threshold.
MAX_GAP = pd.Timedelta("30min")
co2["segment"] = co2.groupby("sensorid")["timestamp"].diff().gt(MAX_GAP).groupby(co2["sensorid"]).cumsum()
period_audit = co2.groupby(["sensorid", "segment"]).agg(
    observations=("timestamp", "size"), start=("timestamp", "min"), end=("timestamp", "max"))
period_audit["duration_days"] = (period_audit["end"] - period_audit["start"]).dt.total_seconds() / 86400
period_audit = period_audit.sort_values(["duration_days", "observations"], ascending=False)
display(period_audit.head(10))
# Selection uses timestamp coverage only, never forecasting error or test values.
selected_sensor, selected_segment = period_audit.index[0]
sensor_series = co2.loc[(co2["sensorid"] == selected_sensor) & (co2["segment"] == selected_segment)].set_index("timestamp").sort_index()
print("Selected sensor:", selected_sensor, "period:", sensor_series.index.min(), "to", sensor_series.index.max())
display(sensor_series[["co2_ppm", "Temperature", "Humidity"]].describe())

In [ ]:
# Optional visual checks; tables and splits do not require matplotlib.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("For time-series and daily-pattern plots, run %pip install matplotlib, then rerun this cell.")
else:
    local_series = sensor_series.copy()
    local_series.index = local_series.index.tz_convert("Australia/Sydney")
    local_series[["co2_ppm", "Temperature", "Humidity"]].plot(subplots=True, figsize=(12, 7), title="Selected sensor: measured values")
    plt.tight_layout()
    plt.show()
    local_series.groupby(local_series.index.hour)[["co2_ppm", "Temperature", "Humidity"]].mean().plot(subplots=True, figsize=(10, 6), title="Mean daily pattern (Sydney local hour)")
    plt.tight_layout()
    plt.show()
    local_series.index.to_series().diff().dt.total_seconds().div(60).hist(bins=30)
    plt.xlabel("Sampling interval (minutes)")
    plt.ylabel("Count")
    plt.show()


## Build forecasts and split chronologically
The selected sensor samples about every 10 minutes. An exact 15-minute measured target is therefore unsupported. Use a **20-minute horizon** with a 2-minute target tolerance, without interpolating targets. Historical CO2 at 10 and 20 minutes ago is matched backward only, allowing 2 minutes of staleness. Compute the rate using actual elapsed time. Review the actual horizon summary below.

Split forecast issue times into approximately 60% training, 20% validation and 20% final test. Purge earlier examples whose target reaches the next period. Historical inputs from a preceding period are available at forecast time and may be used. Tune models on validation only and reserve the final test for the selected configuration. Fit any learned scaling or imputation on training only. Revisit the original 15-minute objective if higher-frequency data becomes available.


In [ ]:
HORIZON = pd.Timedelta("20min")
TOLERANCE = pd.Timedelta("2min")
base = sensor_series[["co2_ppm", "Temperature", "Humidity"]].reset_index()
features = base.rename(columns={"timestamp": "issue_time"}).copy()
for minutes in [10, 20]:
    lookup = base[["timestamp", "co2_ppm"]].rename(columns={"timestamp": f"lag_time_{minutes}", "co2_ppm": f"co2_lag_{minutes}min"})
    features["lookup_time"] = features["issue_time"] - pd.Timedelta(minutes=minutes)
    features = pd.merge_asof(features.sort_values("lookup_time"), lookup,
        left_on="lookup_time", right_on=f"lag_time_{minutes}", direction="backward", tolerance=TOLERANCE)
features = features.drop(columns="lookup_time").sort_values("issue_time")
elapsed_minutes = (features["issue_time"] - features["lag_time_10"]).dt.total_seconds() / 60
features["co2_rate_ppm_per_min"] = (features["co2_ppm"] - features["co2_lag_10min"]) / elapsed_minutes
features["desired_target_time"] = features["issue_time"] + HORIZON
targets = base[["timestamp", "co2_ppm"]].rename(columns={"timestamp": "target_time", "co2_ppm": "target_co2_ppm"})
forecast_data = pd.merge_asof(features.sort_values("desired_target_time"), targets,
    left_on="desired_target_time", right_on="target_time", direction="nearest", tolerance=TOLERANCE)
feature_columns = ["co2_ppm", "co2_lag_10min", "co2_lag_20min", "Temperature", "Humidity", "co2_rate_ppm_per_min"]
rows_before = len(forecast_data)
forecast_data = forecast_data.dropna(subset=feature_columns + ["target_time", "target_co2_ppm"]).reset_index(drop=True)
forecast_data["actual_horizon_minutes"] = (forecast_data["target_time"] - forecast_data["issue_time"]).dt.total_seconds() / 60
print(f"Usable forecasts: {len(forecast_data):,}; excluded for missing inputs/targets: {rows_before - len(forecast_data):,}")
display(forecast_data["actual_horizon_minutes"].describe())
if len(forecast_data) < 100:
    raise ValueError("Too few forecasts. Review sensor coverage, lag intervals and target tolerance.")
validation_start = forecast_data["issue_time"].iloc[int(.6 * len(forecast_data))]
test_start = forecast_data["issue_time"].iloc[int(.8 * len(forecast_data))]
train = forecast_data.loc[(forecast_data.issue_time < validation_start) & (forecast_data.target_time < validation_start)].copy()
validation = forecast_data.loc[(forecast_data.issue_time >= validation_start) & (forecast_data.issue_time < test_start) & (forecast_data.target_time < test_start)].copy()
test = forecast_data.loc[forecast_data.issue_time >= test_start].copy()
assert all(len(part) > 0 for part in [train, validation, test])
assert train.target_time.max() < validation.issue_time.min()
assert validation.target_time.max() < test.issue_time.min()
X_train, y_train = train[feature_columns].copy(), train["target_co2_ppm"].copy()
X_val, y_val = validation[feature_columns].copy(), validation["target_co2_ppm"].copy()
X_test, y_test = test[feature_columns].copy(), test["target_co2_ppm"].copy()
split_summary = pd.DataFrame([{"split": name, "rows": len(part), "start": part.issue_time.min(), "last_issue": part.issue_time.max(), "last_target": part.target_time.max()} for name, part in [("train", train), ("validation", validation), ("test", test)]])
display(split_summary)
print("Purged boundary forecasts:", len(forecast_data) - len(train) - len(validation) - len(test))

In [ ]:
# Establish baseline on training/validation; final test remains reserved.
def persistence_metrics(part):
    before = perf_counter()
    prediction = part["co2_ppm"].to_numpy()
    prediction_seconds = perf_counter() - before
    error = prediction - part["target_co2_ppm"].to_numpy()
    return {"MAE_ppm": np.abs(error).mean(), "RMSE_ppm": np.sqrt(np.mean(error**2)),
            "mean_signed_error_ppm": error.mean(), "prediction_seconds": prediction_seconds}
baseline_results = pd.DataFrame({"train": persistence_metrics(train), "validation": persistence_metrics(validation)}).T
display(baseline_results)
# Export reproducible datasets and audits next to the notebook; raw files stay in place.
output_dir = data_dir.parent / "prepared_data"
output_dir.mkdir(exist_ok=True)
for name, part in [("train", train), ("validation", validation), ("test", test)]:
    part.to_csv(output_dir / f"{name}.csv", index=False)
sensor_audit.to_csv(output_dir / "sensor_audit.csv", index=False)
period_audit.to_csv(output_dir / "period_audit.csv")
split_summary.to_csv(output_dir / "split_summary.csv", index=False)
print("Saved datasets and audit tables to", output_dir)